In [24]:
import pandas as pd
import numpy as np

df_tpm = pd.read_csv("../../rna-seq/star_rsem/rsem.merged.gene_tpm.tsv", sep="\t")
df_tpm.set_index("gene_id", inplace=True)
df_tpm

,transcript_id(s),TGEV_REP1,TGEV_REP2,TGEV_REP3,WT_REP1,WT_REP2,WT_REP3
gene_id,,,,,,,
ENSSSCG00000000002,ENSSSCT00000000003,4.81,3.13,4.73,3.79,5.62,4.07
ENSSSCG00000000003,"ENSSSCT00000000004,ENSSSCT00000052628,ENSSSCT0...",2.19,2.65,3.76,1.81,2.85,4.39
ENSSSCG00000000005,ENSSSCT00000000006,1.60,2.28,1.48,1.03,1.08,0.86
ENSSSCG00000000006,ENSSSCT00000000007,0.91,1.72,1.25,0.88,1.23,1.33
ENSSSCG00000000007,"ENSSSCT00000000008,ENSSSCT00000057594,ENSSSCT0...",8.18,9.08,8.88,5.73,7.24,6.23
...,...,...,...,...,...,...,...
ENSSSCG00000051794,"ENSSSCT00000029953,ENSSSCT00000037685,ENSSSCT0...",15.97,14.91,13.70,9.19,7.75,10.58
ENSSSCG00000051795,ENSSSCT00000075332,0.04,0.03,0.06,1.03,0.88,1.46
ENSSSCG00000051796,ENSSSCT00000083535,0.00,0.00,0.13,0.00,0.00,0.00


In [34]:
df_genePIA = pd.read_csv("PI_100kb_Agene_id")
df_genePIA.columns = ["gene_id"]
df_genePIA.set_index("gene_id", inplace=True)
df_tpmPIA = df_tpm.loc[df_genePIA.index].drop(["transcript_id(s)", "WT_REP1", "WT_REP2", "WT_REP3"], axis=1)
df_tpmPI_A = df_tpmPIA[["TGEV_REP1","TGEV_REP2", "TGEV_REP3"]].mean(axis=1)
df_tpmPI_A

gene_id
ENSSSCG00000000003     2.866667
ENSSSCG00000000005     1.786667
ENSSSCG00000000006     1.293333
ENSSSCG00000000007     8.713333
ENSSSCG00000000010     0.890000
                        ...    
ENSSSCG00000051789     5.340000
ENSSSCG00000051790     0.060000
ENSSSCG00000051792     0.000000
ENSSSCG00000051794    14.860000
ENSSSCG00000051795     0.043333
Length: 20189, dtype: float64

In [26]:
df_genePIB = pd.read_csv("PI_100kb_Bgene_id")
df_genePIB.columns = ["gene_id"]
df_genePIB.set_index("gene_id", inplace=True)
df_tpmPIB = df_tpm.loc[df_genePIB.index].drop(["transcript_id(s)", "WT_REP1", "WT_REP2", "WT_REP3"], axis=1)
df_tpmPI_B = df_tpmPIB[["TGEV_REP1","TGEV_REP2", "TGEV_REP3"]].mean(axis=1)
df_tpmPI_B

gene_id
ENSSSCG00000000295    2.173333
ENSSSCG00000000297    0.030000
ENSSSCG00000000298    0.743333
ENSSSCG00000000299    0.000000
ENSSSCG00000000312    0.000000
                        ...   
ENSSSCG00000051791    0.083333
ENSSSCG00000051793    0.000000
ENSSSCG00000051796    0.043333
ENSSSCG00000051797    0.000000
ENSSSCG00000051798    0.000000
Length: 11109, dtype: float64

In [47]:
df = pd.concat([df_tpmPI_A, df_tpmPI_B], axis=1)
df.columns = ["PI_A", "PI_B"]
df["PI_A"].dropna()

gene_id
ENSSSCG00000000003     2.866667
ENSSSCG00000000005     1.786667
ENSSSCG00000000006     1.293333
ENSSSCG00000000007     8.713333
ENSSSCG00000000010     0.890000
                        ...    
ENSSSCG00000051789     5.340000
ENSSSCG00000051790     0.060000
ENSSSCG00000051792     0.000000
ENSSSCG00000051794    14.860000
ENSSSCG00000051795     0.043333
Name: PI_A, Length: 20189, dtype: float64

In [48]:
df["PI_B"].dropna()

gene_id
ENSSSCG00000000295     2.173333
ENSSSCG00000000363     2.420000
ENSSSCG00000000364    40.210000
ENSSSCG00000000457    29.836667
ENSSSCG00000000464     0.060000
                        ...    
ENSSSCG00000051791     0.083333
ENSSSCG00000051793     0.000000
ENSSSCG00000051796     0.043333
ENSSSCG00000051797     0.000000
ENSSSCG00000051798     0.000000
Name: PI_B, Length: 11109, dtype: float64

In [41]:
df_p = df.melt()
df_p

,variable,value
0,PI_A,2.866667
1,PI_A,1.786667
2,PI_A,1.293333
3,PI_A,8.713333
4,PI_A,0.890000
...,...,...
60615,PI_B,0.083333
60616,PI_B,0.000000
60617,PI_B,0.043333
60618,PI_B,0.000000


In [43]:
df_p.columns = ["Type", "Tpm"]
df_p['Tpm'] = df_p['Tpm'].apply(lambda x: np.log(x + 0.00001))
df_p

,Type,Tpm
0,PI_A,0.051795
1,PI_A,-0.544104
2,PI_A,-1.357774
3,PI_A,0.772358
4,PI_A,NaN
...,...,...
60615,PI_B,NaN
60616,PI_B,NaN
60617,PI_B,NaN
60618,PI_B,NaN


In [54]:
from scipy.stats import ranksums
z, p = ranksums(df['PI_A'].dropna(), df['PI_B'].dropna())
print(z, p)

55.93685032800369 0.0


In [56]:
import plotly.express as px
import plotly.io as pio
import numpy as np

fig = px.box(df_p.dropna(), 
             x="Type",
             y="Tpm", 
             color="Type",
             notched=True
            )

fig.update_layout(
    width=500,
    height=500,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

# 添加标题，并设置标题的位置
fig.update_layout(
    title={
        'text': "A Vs B",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    }
)


fig.update_yaxes(range=[-8, 5])

# 将图像保存为PDF文件
pio.write_image(fig, 'PIboxplot.pdf')
fig.show()